# The M5 Forecasting Engine

In [1]:
# ==============================================================================
# CELL 1: ARCHITECTURE SETUP & CONFIGURATION
# ==============================================================================
# --- 1. CORE IMPORTS ---
from __future__ import annotations
import os
import shutil
import zipfile
import gc
from pathlib import Path
import warnings
import random

warnings.filterwarnings("ignore")


# --- 2. Centralized Configuration (Single Source of Truth) ---
class Config:
    """
    Immutable configuration parameters for the pipeline.
    """
    # Reproducibility
    SEED: int = 75

    # Path Management
    BASE_DIR: Path = Path.cwd().parent
    DATA_DIR: Path = BASE_DIR / "data" / "m5"
    TRAIN_FILE: Path = "sales_train_validation.csv"
    CALENDAR_FILE: Path = "calendar.csv"
    PRICES_FILE: Path = "sell_prices.csv"

    # Optimization Constants
    N_FOLDS = 5
    # Default target for M5 is predicting the sales volume
    TARGET_COL: str = "sales"


# --- 3. Deterministic Environment Lock ---
os.environ["PYTHONHASHSEED"] = str(Config.SEED)
random.seed(Config.SEED)

# --- 4. Data ---
import numpy as np

np.random.seed(Config.SEED)

# External Frameworks & Display Engines
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display & Plot Configuration
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1000)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

%matplotlib inline
%config InlineBackend.figure_format = "retina"
plt.rcParams.update(
        {"figure.figsize": (12, 5), "figure.dpi": 100, 'figure.max_open_warning': 0, "font.size": 11, "axes.grid": True,
                "grid.alpha": 0.3, "axes.titlesize": 13, "axes.labelsize": 11, "axes.unicode_minus": False, })
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)

print("[*] Core Library Imported Successfully. Environment Configured.")

[*] Core Library Imported Successfully. Environment Configured.


In [2]:
# =============================================================================
# CELL 2: KAGGLE AUTHENTICATION, DOWNLOAD, AND MEM-OPT INGESTION
# =============================================================================

# Use the API directly instead of subprocess
from kaggle.api.kaggle_api_extended import KaggleApi

def setup_kaggle_credentials():
    """
    Locates kaggle.json in the current working directory and safely
    copies it to the WSL/Linux home directory with strict 0o600 permissions.
    """
    print("--- Setting up Kaggle Credentials ---")
    current_dir_key = Path("kaggle.json")
    home_dir_kaggle = Path.home() / ".kaggle"
    dest_key = home_dir_kaggle / "kaggle.json"

    # Create ~/.kaggle directory if it doesn't exist
    if not home_dir_kaggle.exists():
        home_dir_kaggle.mkdir(parents=True)
        print(f"[*] Created directory: {home_dir_kaggle}")

    # Copy the key and set strict permissions
    if current_dir_key.exists():
        shutil.copy(current_dir_key, dest_key)
        os.chmod(dest_key, 0o600)
        print(f"[*] Copied kaggle.json to {dest_key} and set permissions to 600.")
    elif dest_key.exists():
        print(f"[*] kaggle.json already exists in {dest_key}. Proceeding.")
    else:
        raise FileNotFoundError(
            f"Could not find 'kaggle.json' in the current directory ({Path.cwd()}) "
            f"or in {home_dir_kaggle}. Please place it in the project root."
        )

def fetch_m5_data_via_api(download_path: Path):
    """
    Authenticates with Kaggle API and downloads the M5 competition dataset.
    """
    setup_kaggle_credentials()

    print("\n--- Authenticating with Kaggle ---")
    api = KaggleApi()
    api.authenticate()
    print("[*] Authentication Successful!")

    if not download_path.exists():
        download_path.mkdir(parents=True, exist_ok=True)

    competition_name = "m5-forecasting-accuracy"
    expected_file = download_path / "sales_train_validation.csv"

    if expected_file.exists():
        print("[*] M5 Data already exists in target directory. Skipping download.")
        return

    print(f"\n--- Downloading M5 Competition Data ---")
    print(f"[*] Downloading to: {download_path} ... This may take a few minutes.")

    # Download the entire competition bundle
    api.competition_download_files(competition_name, path=str(download_path))

    zip_path = download_path / f"{competition_name}.zip"
    if zip_path.exists():
        print(f"[*] Extracting {zip_path.name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(download_path)
        zip_path.unlink() # Cleanup zip
        print("[*] Extraction complete and zip removed.")
    else:
        print("[!] Warning: Expected zip file not found after download.")

# === MEM-OPT ===

def reduce_mem_usage(df: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    """
    Iterates through all columns of a dataframe and modifies the data type
    to reduce memory usage safely based on min/max values.
    """
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    start_mem = df.memory_usage().sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtypes

        if col_type in numerics:
            c_min = df[col].min()
            c_max = df[col].max()

            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)

    end_mem = df.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory decreased from {start_mem:.2f} MB to {end_mem:.2f} MB ({(100 * (start_mem - end_mem) / start_mem):.1f}% reduction)")

    return df

In [3]:
# =============================================================================
# CELL 3: PRODUCTION INGESTION & SCHEMA ENFORCEMENT
# =============================================================================
from supply_chain_pred_core import SemanticSchemaRegistry, FeatureContract, ColumnRole

def ingest_and_enforce_schema(data_dir: Path, target_store: str = 'CA_1'):
    """
    1. Loads raw wide-format data and filters to target store.
    2. Melts to long format.
    3. Coerces object types to category (LightGBM prerequisite).
    4. Passes the result through the core SemanticSchemaRegistry to lock the schema.
    """
    print(f"\n--- 1. Raw Ingestion (Store: {target_store}) ---")
    sales_file = data_dir / Config.TRAIN_FILE
    df_wide = pd.read_csv(sales_file)

    # Filter to POC store to keep iteration fast (5.8M rows is plenty for signal detection)
    df_wide = df_wide[df_wide['store_id'] == target_store].copy()

    print("--- 2. Unpivoting (Melt) ---")
    id_vars = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
    df_long = pd.melt(df_wide, id_vars=id_vars, var_name='d', value_name='sales')

    del df_wide
    gc.collect()

    print("--- 3. Type Coercion (Object to Category) ---")
    for col in df_long.columns:
        if df_long[col].dtype == 'object':
            df_long[col] = df_long[col].astype('category')

    print("--- 4. Semantic Schema Registry (Core Library) ---")
    # Initialize the core registry in strict mode
    registry = SemanticSchemaRegistry(strict_mode=True)

    # Define physical expectations for our data
    contracts = [
        FeatureContract(
            name="sales", dependencies=[], dtype_family="numeric",
            role=ColumnRole.TARGET, min_val=0.0 # Physical bound: Cannot sell negative units
        ),
        FeatureContract(
            name="item_id", dependencies=[], dtype_family="categorical",
            role=ColumnRole.IDENTITY
        ),
        FeatureContract(
            name="store_id", dependencies=[], dtype_family="categorical",
            role=ColumnRole.IDENTITY
        )
    ]

    # Register the rules
    registry.register_contracts(contracts)

    # Validate the data against our rules (e.g., checks if any sales are < 0)
    registry.validate_semantic_bounds(df_long)

    # Baseline the physical schema memory to prevent drift in future pipeline steps
    registry.capture_checkpoint(df_long, "post_ingestion_base")

    return df_long, registry

# Execute the secured ingestion pipeline
df_sales_long, global_registry = ingest_and_enforce_schema(Config.DATA_DIR)
print(f"\n[*] Secured Pipeline Ready. Matrix Shape: {df_sales_long.shape}")


--- 1. Raw Ingestion (Store: CA_1) ---
--- 2. Unpivoting (Melt) ---
--- 3. Type Coercion (Object to Category) ---


[SemanticRegistry] INFO: Registered 3 explicit feature contracts.
[SemanticRegistry] INFO: All features successfully passed semantic bound validation.
[SemanticRegistry] INFO: Captured reactive checkpoint 'post_ingestion_base' | Features: 8


--- 4. Semantic Schema Registry (Core Library) ---

[*] Secured Pipeline Ready. Matrix Shape: (5832737, 8)


In [4]:
# =============================================================================
# CELL 4: ARCHITECTURAL FEATURE ENGINEERING & TIME-SERIES SPLIT
# =============================================================================

def build_production_features_and_split(df_long: pd.DataFrame, data_dir: Path):
    print("--- 1. Extracting Temporal Engine ---")
    df_long['d_int'] = df_long['d'].astype(str).str.replace('d_', '').astype(np.int16)

    print("--- 2. Denormalization (Contextual Envelope) ---")
    # Calendar
    calendar = pd.read_csv(data_dir / Config.CALENDAR_FILE)
    cols_cal = ['d', 'wm_yr_wk', 'snap_CA', 'snap_TX', 'snap_WI']
    calendar = calendar[cols_cal]
    # Downcast calendar safely
    for col in ['snap_CA', 'snap_TX', 'snap_WI']:
        calendar[col] = calendar[col].astype(np.int8)

    df_long = df_long.merge(calendar, on='d', how='left')
    del calendar; gc.collect()

    # Prices
    prices = pd.read_csv(data_dir / Config.PRICES_FILE)
    prices['sell_price'] = prices['sell_price'].astype(np.float32)
    df_long = df_long.merge(prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
    del prices; gc.collect()

    print("--- 3. Longitudinal Firewall (Lags & Momentum) ---")
    # Strict Sort to prevent Look-Ahead Bias
    df_long = df_long.sort_values(['store_id', 'item_id', 'd_int']).reset_index(drop=True)

    # Fast grouping for shifted features
    grouped = df_long.groupby(['store_id', 'item_id'])['sales']

    df_long['lag_7'] = grouped.shift(7).astype(np.float32)
    df_long['lag_28'] = grouped.shift(28).astype(np.float32)

    # Rolling momentum (Must apply shift BEFORE rolling!)
    df_long['rmean_7_28'] = grouped.transform(lambda x: x.shift(28).rolling(7).mean()).astype(np.float32)

    # Cyclicality
    df_long['day_of_week'] = (df_long['d_int'] % 7).astype('category')

    # Drop rows with NaNs from the lag look-back period
    df_clean = df_long.dropna().copy()

    print("--- 4. Strict Time-Series Walk-Forward Split ---")
    max_day = df_clean['d_int'].max()
    split_day = max_day - 28 # Hold out the last 28 days for validation

    train_mask = df_clean['d_int'] <= split_day
    val_mask = df_clean['d_int'] > split_day

    df_train = df_clean[train_mask]
    df_val = df_clean[val_mask]

    print(f"[*] Train Shape: {df_train.shape} | Days: {df_train['d_int'].min()} to {df_train['d_int'].max()}")
    print(f"[*] Val Shape:   {df_val.shape}  | Days: {df_val['d_int'].min()} to {df_val['d_int'].max()}")

    # Define Final Features
    exclude = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'd_int']
    features = [c for c in df_clean.columns if c not in exclude]

    print(f"[*] Engineered Features: {features}")

    return df_train, df_val, features

# Execute the pipeline
df_train, df_val, model_features = build_production_features_and_split(df_sales_long, Config.DATA_DIR)

--- 1. Extracting Temporal Engine ---
--- 2. Denormalization (Contextual Envelope) ---
--- 3. Longitudinal Firewall (Lags & Momentum) ---
--- 4. Strict Time-Series Walk-Forward Split ---
[*] Train Shape: (4572127, 18) | Days: 35 to 1885
[*] Val Shape:   (85372, 18)  | Days: 1886 to 1913
[*] Engineered Features: ['wm_yr_wk', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'lag_7', 'lag_28', 'rmean_7_28', 'day_of_week']


In [5]:
# =============================================================================
# CELL 5: GENERIC PRODUCTION TRAINING & FINAL EVALUATION
# =============================================================================

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error

from supply_chain.supply_chain_pred_core import SupplyChainModelTrainer


# 1. Define the specific evaluation metric for M5
def m5_rmse_eval(preds: np.ndarray, val_data: any):
    labels = val_data.get_label()
    # Physical constraint: Inventory cannot be negative
    preds = np.clip(preds, 0, None)
    rmse = np.sqrt(mean_squared_error(labels, preds))
    return [('RMSE', rmse, False)]

# 2. Define the Custom Search Space explicitly for this project
def m5_tweedie_search_space(trial):
    return {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 128),
        'tweedie_variance_power': trial.suggest_float('tweedie_variance_power', 1.1, 1.7),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.9),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100)
    }

# 3. Initialize the Core Trainer (Your generic class)
print("--- 1. Initializing Generic OOP Trainer ---")
trainer = SupplyChainModelTrainer(
    objective_func='tweedie',
    eval_metric_func=m5_rmse_eval,
    is_time_series=True,
    n_splits=3
)

# 4. Optuna Optimization via Dependency Injection
print("\n--- 2. Launching Core Optimization (Injected Search Space) ---")

# THE FIX: Only pass categorical columns that actually exist in df_train[model_features]
cat_cols = ['day_of_week']
dummy_weights = pd.Series(1.0, index=df_train.index)

# Run optimization to find the best parameters
best_params = trainer.optimize_hyperparameters(
    X=df_train[model_features],
    y=df_train['sales'],
    weights=dummy_weights,
    categorical_cols=cat_cols,
    n_trials=10,
    custom_search_space=m5_tweedie_search_space,
    pruning_metric='RMSE'
)

print(f"\n[*] Optuna Complete. Winning Parameters: {best_params}")

# 5. Native Final Training (Since the core handles CV/Tuning)
print("\n--- 3. Training Final Production Model ---")
# Create LightGBM datasets natively
dtrain_final = lgb.Dataset(df_train[model_features], label=df_train['sales'], categorical_feature=cat_cols)
dval_final = lgb.Dataset(df_val[model_features], label=df_val['sales'], reference=dtrain_final)

# Ensure objective and metric are explicitly set for the final run
final_params = best_params.copy()
final_params['objective'] = 'tweedie'
final_params['metric'] = 'rmse'
final_params['verbose'] = -1

# Train the master model
final_model = lgb.train(
    final_params,
    dtrain_final,
    num_boost_round=600,
    valid_sets=[dtrain_final, dval_final],
    callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)]
)

# 6. Native Evaluation on Holdout (The Final Test)
print("\n--- 4. Final Evaluation (Holdout Test) ---")
final_preds = final_model.predict(df_val[model_features]).clip(0)
final_rmse = np.sqrt(mean_squared_error(df_val['sales'], final_preds))

print(f"\n======================================")
print(f"[*] FINAL HOLDOUT RMSE: {final_rmse:.4f}")
print(f"======================================")

[SC_Core] INFO: [*] Launching Optuna Bayesian Optimization...


--- 1. Initializing Generic OOP Trainer ---

--- 2. Launching Core Optimization (Injected Search Space) ---


Optuna Trials:   0%|          | 0/10 [00:00<?, ?it/s]

[SC_Core] INFO: CV Routing: Using TimeSeriesSplit (Preventing Future Leakage)
[SC_Core] INFO: CV Routing: Using TimeSeriesSplit (Preventing Future Leakage)
[SC_Core] INFO: CV Routing: Using TimeSeriesSplit (Preventing Future Leakage)
[SC_Core] INFO: CV Routing: Using TimeSeriesSplit (Preventing Future Leakage)
[SC_Core] INFO: CV Routing: Using TimeSeriesSplit (Preventing Future Leakage)
[SC_Core] INFO: CV Routing: Using TimeSeriesSplit (Preventing Future Leakage)
[SC_Core] INFO: CV Routing: Using TimeSeriesSplit (Preventing Future Leakage)
[SC_Core] INFO: CV Routing: Using TimeSeriesSplit (Preventing Future Leakage)
[SC_Core] INFO: CV Routing: Using TimeSeriesSplit (Preventing Future Leakage)
[SC_Core] INFO: CV Routing: Using TimeSeriesSplit (Preventing Future Leakage)
[SC_Core] INFO: [*] Optimization Complete. Best OOF Metric: 2.4112
[SC_Core] INFO: [*] Best Params:
{'learning_rate': 0.023688639503640783, 'num_leaves': 124, 'tweedie_variance_power': 1.539196365086843, 'feature_fractio


[*] Optuna Complete. Winning Parameters: {'learning_rate': 0.023688639503640783, 'num_leaves': 124, 'tweedie_variance_power': 1.539196365086843, 'feature_fraction': 0.7394633936788146, 'min_child_samples': 19, 'objective': 'tweedie', 'metric': 'binary_logloss', 'random_state': 42}

--- 3. Training Final Production Model ---

--- 4. Final Evaluation (Holdout Test) ---

[*] FINAL HOLDOUT RMSE: 2.1736


In [6]:
# =============================================================================
# CELL 6: DECISION INTELLIGENCE - CONFORMAL REGRESSION BUFFER
# =============================================================================

from supply_chain.supply_chain_pred_core import ForecastingConformalEngine

# Execution Pipeline

print("--- 1. Calibrating Forecasting Risk Engine (Gate 10) ---")

# Initialize our new regression-adapted engine
risk_engine = ForecastingConformalEngine(alpha=0.05) # 95% SLA

# Get predictions on Validation set to learn the error distribution
val_preds = final_model.predict(df_val[model_features]).clip(0)
tau = risk_engine.calibrate(y_true=df_val['sales'], y_pred=val_preds)

print(f"[*] Calibration Complete.")
print(f"[*] Conformal Uncertainty Bound (Tau_hat): {tau:.2f} Units.")
print(f"    (Meaning: 95% of the time, our model's error is {tau:.2f} units or less).")

print("\n--- 2. Generating Risk-Aware Procurement Recommendations ---")

# Simulating Holdout/Test behavior
df_val_final = df_val.copy()
holdout_preds = final_model.predict(df_val_final[model_features]).clip(0)

# Mocking business reality: Lead time of 2 weeks
df_val_final['lead_time_weeks'] = 2.0

recommendations = risk_engine.calculate_safety_buffer(
    base_forecasts=pd.Series(holdout_preds, index=df_val_final.index),
    lead_time_weeks=df_val_final['lead_time_weeks']
)

# Merge back with item ID and actuals to inspect results
results_view = pd.concat([
    df_val_final[['item_id', 'sales']],
    recommendations
], axis=1).reset_index(drop=True)

print("\n[*] Sample Recommendations (Decision Intelligence):")
display(results_view.head(10))

# --- THE EXECUTIVE BOTTOM LINE ---
total_buffer_needed = recommendations['Safety_Buffer_Units'].sum()
print(f"\n========================================================")
print(f"[*] SYSTEM DEPLOYMENT READY.")
print(f"[*] To guarantee a 95% SLA across these SKUs over a 2-week lead time,")
print(f"[*] Company must allocate a total safety buffer of: {total_buffer_needed:,.0f} units.")
print(f"========================================================")

--- 1. Calibrating Forecasting Risk Engine (Gate 10) ---
[*] Calibration Complete.
[*] Conformal Uncertainty Bound (Tau_hat): 3.74 Units.
    (Meaning: 95% of the time, our model's error is 3.74 units or less).

--- 2. Generating Risk-Aware Procurement Recommendations ---

[*] Sample Recommendations (Decision Intelligence):


,item_id,sales,Base_Forecast,Safety_Buffer_Units,Total_Required_Inventory
0,FOODS_1_001,2,0.4225,6.0000,7.0000
1,FOODS_1_001,1,0.6942,6.0000,7.0000
2,FOODS_1_001,1,0.7477,6.0000,7.0000
3,FOODS_1_001,0,0.3687,6.0000,7.0000
4,FOODS_1_001,4,0.6419,6.0000,7.0000
5,FOODS_1_001,0,1.0233,6.0000,8.0000
6,FOODS_1_001,0,0.6546,6.0000,7.0000
7,FOODS_1_001,4,0.8904,6.0000,7.0000
8,FOODS_1_001,1,0.6538,6.0000,7.0000
9,FOODS_1_001,3,0.5505,6.0000,7.0000



[*] SYSTEM DEPLOYMENT READY.
[*] To guarantee a 95% SLA across these SKUs over a 2-week lead time,
[*] Company must allocate a total safety buffer of: 512,232 units.
